# Reference pose for the sensor-constrained agent (`mobile_2`)

`mobile_2` carries a RealSense D455 and Isaac VSLAM. No LiDAR. This notebook derives its
reference trajectory in the **frozen GLIM map frame** three ways, so the arms can be compared:

| Arm | Constraints | What it tests |
|-----|-------------|---------------|
| **A** | depth-cloud -> map registration only | can frustum geometry alone hold the pose? |
| **B** | VSLAM odometry + fiducial board sightings | do sparse absolute anchors bound the drift? |
| **C** | all of the above, jointly | does each fix the other's failure mode? |

Evaluation is **leave-one-board-out**: estimate with two boards, score on the third.

**Run order.** Cells are independent and cache to `WORK/`. Re-running a cell reads the cache
unless you set its `FORCE = True`. Steps 0a-0c are gates: if they fail, later arms are not
meaningful, so read their output before continuing.

**Frame convention used throughout**
- `map` - the GLIM reference map frame (the trajectory frame of `01a_refine_poses.py`)
- `N` - the normalised/board frame from the survey JSON (`T_N_world`), anchor board at origin
- the graph pose variable is the **infra1 optical frame** of `mobile_2` (depth is registered
  to infra1, so the depth cloud lives in this frame with no extra extrinsic)
- pose perturbation: `t <- t + dt` (world), `R <- R @ exp(dphi)` (body). Used consistently in
  the ICP solver and the pose graph so no Jacobian conversion is ever needed.

In [ ]:
# ============================== CONFIG ==============================
from pathlib import Path

# --- inputs -------------------------------------------------------------
BAG        = Path("/home/wicoms-robot/data/mirc_dataset_coop2_20260828_merged")   # rosbag2 dir (has metadata.yaml)
SURVEY     = Path("/home/wicoms-robot/workspaces/isaac_ros-dev/map_stages_20260828/board_survey.json")
REF_MAP    = Path("/home/wicoms-robot/workspaces/isaac_ros-dev/map_stages_20260828/map_refined.ply")
M1_TRAJ    = Path("/home/wicoms-robot/workspaces/isaac_ros-dev/map_stages_20260828/traj_lidar_refined_r3.txt")  # mobile_1, TUM, map frame (for step 0c only)

WORK       = Path("./m2_reference"); WORK.mkdir(exist_ok=True)

# --- topics -------------------------------------------------------------
T_INFRA1     = "/mobile_2/infra1/image_rect_raw"
T_INFRA1_CI  = "/mobile_2/infra1/camera_info"
T_DEPTH      = "/mobile_2/depth/image_rect_raw"
T_DEPTH_CI   = "/mobile_2/depth/camera_info"
T_COLOR      = "/mobile_2/color/image_raw"
T_COLOR_CI   = "/mobile_2/color/camera_info"
T_VO         = "/mobile_2/visual_slam/tracking/odometry"
T_M1_IMG     = "/mobile_1/zed/left/image_rect_color"
T_M1_CI      = "/mobile_1/zed/left/camera_info"

# --- step 0b: sighting census ------------------------------------------
CENSUS_STRIDE   = 2        # detect every Nth infra1 frame (27.5 Hz -> ~14 Hz)
MIN_CORNERS     = 6        # ChArUco corners needed to accept a sighting
MAX_REPROJ_PX   = 1.0      # reject a sighting above this mean reprojection error
MARKER_RATIO    = 0.75     # marker_len / square_len; not in the survey JSON, detection-only

# --- step 1: depth -> cloud --------------------------------------------
DEPTH_SCALE     = 0.001    # 16UC1 -> metres (RealSense default)
RANGE_MIN       = 0.40
RANGE_MAX       = 3.50     # D455 error ~2% of z; 3.5 m keeps it under the 2.9 cm map noise floor
EDGE_JUMP       = 0.05     # reject pixels with >5 cm depth step in a 3x3 -> kills flying pixels
CLOUD_VOXEL     = 0.05     # voxel CENTROID (never voxel centre)
REG_RATE_HZ     = 10.0     # registration / graph rate
MAX_PTS_PER_FR  = 8000     # random subsample after voxelising, caps ICP cost

# --- step 2: reference map ---------------------------------------------
TARGET_VOXEL    = 0.05     # map downsample for the KD-tree
PLANE_VOXEL     = 0.40     # patch size for local plane fitting
NORMAL_K        = 20

# --- arm A: ICP ---------------------------------------------------------
MAX_CORR        = [0.40, 0.20, 0.10]
ITERS_PER_GATE  = 5
HUBER           = 0.05
PRIOR_BETA      = 0.10     # >0.05 used for LiDAR: an 87 deg frustum is far more degenerate
DEGEN_RATIO     = 0.02     # eigenvalue < ratio * max(eigenvalue) => direction unobservable

# --- arms B/C: pose graph ----------------------------------------------
VO_SIGMA_T      = 0.002    # m   per 0.1 s of VSLAM relative motion  } MEASURED in the
VO_SIGMA_R      = 0.0005   # rad per 0.1 s of VSLAM relative motion  } "VSLAM noise" cell
PIX_SIGMA       = 0.5      # px, ChArUco corner
ESTIMATE_VO_SCALE = True   # 7-17 m board baselines make VSLAM stereo scale observable
GN_ITERS        = 12

LIMIT_FRAMES    = None     # set e.g. 300 while developing, None for the full bag
print("config loaded ->", WORK.resolve())

In [ ]:
# ============================== IMPORTS + SE(3) ==============================
import json, math, sys, time
import numpy as np
from scipy.spatial import cKDTree
from scipy.spatial.transform import Rotation as Rot
from scipy import ndimage, sparse
from scipy.sparse.linalg import spsolve
import matplotlib; import matplotlib.pyplot as plt

def hat(v):
    x, y, z = v
    return np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])

def Rt(R, t):
    T = np.eye(4); T[:3, :3] = R; T[:3, 3] = t; return T

def inv(T):
    R = T[:3, :3]; o = np.eye(4); o[:3, :3] = R.T; o[:3, 3] = -R.T @ T[:3, 3]; return o

def q_to_R(q):        # q = xyzw
    return Rot.from_quat(q).as_matrix()

def R_to_q(R):
    return Rot.from_matrix(R).as_quat()

def log_R(R):
    return Rot.from_matrix(R).as_rotvec()

def exp_r(w):
    return Rot.from_rotvec(w).as_matrix()

def jr_inv(w):
    """Inverse right Jacobian of SO(3). Exact - NOT the small-angle approximation.
    Pose-graph rotation residuals are large at the first iteration, and J_r^-1 ~ I there
    is wrong by O(20%), which stalls Gauss-Newton instead of converging it."""
    th = float(np.linalg.norm(w)); W = hat(w)
    if th < 1e-6: return np.eye(3) + 0.5 * W
    a = 1.0 / th ** 2 - (1.0 + math.cos(th)) / (2.0 * th * math.sin(th))
    return np.eye(3) + 0.5 * W + a * (W @ W)

def apply(T, P):      # P (N,3)
    return P @ T[:3, :3].T + T[:3, 3]

# --- TUM trajectory io (round-trip exact) -------------------------------
def load_traj(path):
    """-> (t (N,), T (N,4,4))"""
    a = np.loadtxt(path)
    if a.ndim == 1: a = a[None, :]
    T = np.tile(np.eye(4), (len(a), 1, 1))
    T[:, :3, :3] = Rot.from_quat(a[:, 4:8]).as_matrix()
    T[:, :3, 3] = a[:, 1:4]
    return a[:, 0].copy(), T

def write_traj(path, ts, Ts):
    q = Rot.from_matrix(Ts[:, :3, :3]).as_quat()
    out = np.column_stack([ts, Ts[:, :3, 3], q])
    np.savetxt(path, out, fmt="%.9f " + " ".join(["%.9f"] * 7))
    print("wrote", path, out.shape)

def interp_traj(ts_src, Ts_src, ts_q):
    """SLERP + linear interpolation onto query stamps; clamps at the ends."""
    ts_q = np.clip(ts_q, ts_src[0], ts_src[-1])
    i = np.clip(np.searchsorted(ts_src, ts_q) - 1, 0, len(ts_src) - 2)
    d = (ts_src[i + 1] - ts_src[i]); a = np.where(d > 0, (ts_q - ts_src[i]) / np.where(d > 0, d, 1), 0.0)
    from scipy.spatial.transform import Slerp
    key = Rot.from_matrix(Ts_src[:, :3, :3])
    sl = Slerp(ts_src, key)
    R = sl(ts_q).as_matrix()
    t = Ts_src[i, :3, 3] * (1 - a)[:, None] + Ts_src[i + 1, :3, 3] * a[:, None]
    out = np.tile(np.eye(4), (len(ts_q), 1, 1)); out[:, :3, :3] = R; out[:, :3, 3] = t
    return out

print("numpy", np.__version__)
try:
    import cv2; print("opencv", cv2.__version__)
except ImportError:
    cv2 = None; print("!! opencv missing - steps 0b/0c will not run")

## Step 0a - board survey -> map frame

The survey JSON stores board poses in the **normalised frame `N`** (`T_N_world`, anchor board at
the origin, a 3.11 deg yaw off the map frame). The GLIM trajectory lives in the **map** frame, so
every board pose is pulled back through `inv(T_N_world)`.

Two things this cell also does, both load-bearing:

1. **Assigns per-board sigmas as `max(section std, loop-closure disagreement)`.** The raw
   `std_mm` is optimistic for boards with a thin second section. `anchor_b` reports 2.01 mm std
   but carries `drift_warning: true` and a *significant* 12.94 mm / 2.64 deg loop closure
   (ratio 4.24) - its second section has n=5 views. Weighting all three boards equally is the
   mistake that makes the ablation look good and then fails the hold-out test.
2. **Flags the ID collision.** `anchor` and `anchor_b` are the *same physical design* -
   DICT_4X4_50, 9x7, 20 mm, `id_offset: 0`. Detection cannot tell them apart by marker ID; they
   must be disambiguated by position against a pose prior. Handled in the census cell.

In [ ]:
# ============================== STEP 0a: board survey ==============================
S = json.loads(SURVEY.read_text())
T_N_world = np.array(S["T_N_world"]); T_world_N = inv(T_N_world)

BOARDS = {}
for name, b in S["boards"].items():
    T_N_b = Rt(q_to_R(b["qxyzw"]), np.array(b["xyz"]))
    lc = b.get("loop_closure", {}) or {}
    sig_t = max(b.get("std_mm", 0.0), lc.get("mm", 0.0)) * 1e-3        # metres
    sig_r = math.radians(max(lc.get("deg", 0.0), 0.2))                 # rad, floor at 0.2 deg
    BOARDS[name] = dict(
        name=name,
        T_map_board=T_world_N @ T_N_b,
        squares=tuple(b["squares"]), square_len=b["square_len"],
        dictionary=b["dictionary"], id_offset=b.get("id_offset", 0),
        sigma_t=sig_t, sigma_r=sig_r,
        n_views=b.get("n_views", 0), std_mm=b.get("std_mm", np.nan),
        drift_warning=bool(b.get("drift_warning", False)),
        lc_significant=bool(lc.get("significant", False)),
    )

print(f"{'board':11s} {'x':>8s} {'y':>8s} {'z':>8s}  {'sig_t':>7s} {'sig_R':>7s} {'views':>5s}  flags")
for n, b in BOARDS.items():
    p = b["T_map_board"][:3, 3]
    fl = ",".join([f for f, on in [("DRIFT", b["drift_warning"]),
                                   ("LC-SIG", b["lc_significant"])] if on]) or "-"
    print(f"{n:11s} {p[0]:8.3f} {p[1]:8.3f} {p[2]:8.3f}  "
          f"{b['sigma_t']*1000:6.1f}mm {math.degrees(b['sigma_r']):6.2f}d {b['n_views']:5d}  {fl}")

# ID collisions: same dictionary + same id_offset => indistinguishable by marker id
from collections import defaultdict
grp = defaultdict(list)
for n, b in BOARDS.items(): grp[(b["dictionary"], b["id_offset"], b["squares"])].append(n)
AMBIG = {k: v for k, v in grp.items() if len(v) > 1}
for k, v in AMBIG.items():
    print(f"\n!! ID COLLISION {v} share {k[0]} offset {k[1]} {k[2]} "
          "-> disambiguate by position, not id")

# board-board baselines: free long-baseline control distances for the rangefinder
names = list(BOARDS)
print("\nsurveyed baselines (shoot these with the rangefinder - long, one in a corridor):")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        d = np.linalg.norm(BOARDS[names[i]]["T_map_board"][:3, 3] -
                           BOARDS[names[j]]["T_map_board"][:3, 3])
        print(f"  {names[i]:11s} -> {names[j]:11s} {d:7.3f} m")

## Step 0b - sighting census  **(GATE)**

Everything downstream depends on one unknown: **how often does `mobile_2` actually see a board?**

Decision rule after this cell:
- **>= 4 well-separated sighting windows** across the trajectory -> the three-arm ablation is real.
- **1-2 windows, clustered** -> arm B is "VSLAM with one anchor". Say so and reframe the paper
  claim as *anchored vs unanchored VSLAM*, not a fusion ablation.

**Detect in `infra1`, not `color`.** infra1 is global shutter (color is rolling - at 1 m/s a
board is skewed), it is factory-rectified, and depth is already registered to it, so the board
pose lands in the depth frame with no extra extrinsic. The one risk is the IR projector pattern
sitting on the board: watch `mean_reproj_px` against the survey's 0.22-0.39 px. If it is much
worse, flip `USE_COLOR = True` and carry `T_infra1_color`.

In [ ]:
# ============================== bag reader ==============================
import rosbag2_py
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utility import get_message

def bag_reader(path):
    r = rosbag2_py.SequentialReader()
    r.open(rosbag2_py.StorageOptions(uri=str(path), storage_id="mcap"),
           rosbag2_py.ConverterOptions("", ""))
    types = {t.name: t.type for t in r.get_all_topics_and_types()}
    return r, types

def iter_topic(path, topic, stride=1, limit=None):
    """Yield (t_sec, msg). t_sec is the message header stamp when present, else bag time."""
    r, types = bag_reader(path)
    if topic not in types: raise KeyError(f"{topic} not in bag; have {sorted(types)[:8]}...")
    cls = get_message(types[topic])
    f = rosbag2_py.StorageFilter(); f.topics = [topic]; r.set_filter(f)
    i = n = 0
    while r.has_next():
        _, data, t_bag = r.read_next()
        if i % stride: i += 1; continue
        i += 1
        m = deserialize_message(data, cls)
        h = getattr(m, "header", None)
        t = (h.stamp.sec + h.stamp.nanosec * 1e-9) if h is not None else t_bag * 1e-9
        yield t, m
        n += 1
        if limit and n >= limit: break

def first_msg(path, topic):
    for t, m in iter_topic(path, topic, limit=1): return m
    return None

def camera_K(path, topic_ci):
    ci = first_msg(path, topic_ci)
    K = np.array(ci.k).reshape(3, 3)
    D = np.array(ci.d, dtype=float)
    # image_rect_raw / rect_color are already rectified -> D must be treated as zero
    return K, np.zeros(5), ci.width, ci.height

def img_to_np(m):
    a = np.frombuffer(m.data, dtype=np.uint8)
    enc = m.encoding
    if enc in ("mono8", "8UC1"):   return a.reshape(m.height, m.step)[:, :m.width]
    if enc == "16UC1":             return a.view(np.uint16).reshape(m.height, m.step // 2)[:, :m.width]
    ch = {"rgb8": 3, "bgr8": 3, "rgba8": 4, "bgra8": 4}[enc]
    im = a.reshape(m.height, m.step // ch, ch)[:, :m.width]
    return im[..., ::-1][..., :3] if enc.startswith("rgb") else im[..., :3]

def odom_to_T(m):
    p = m.pose.pose.position; o = m.pose.pose.orientation
    return Rt(q_to_R([o.x, o.y, o.z, o.w]), np.array([p.x, p.y, p.z]))

K_IR, _, W_IR, H_IR = camera_K(BAG, T_INFRA1_CI)
print("infra1 K\n", K_IR, f"\n{W_IR}x{H_IR}")

In [ ]:
# ============================== ChArUco detection ==============================
# The board-frame axis convention is NOT fully determined by the survey JSON
# (board_origin="center", board_axes="ros" name a convention without defining its rotation).
# It CANNOT be recovered from detections alone: PnP under any rigid candidate reproduces the
# same image corners and the same board ORIGIN - only the board's orientation frame changes.
# The check cell below picks it from board-to-board RELATIVE rotations against the survey;
# a wrong convention is off by ~90-180 deg there, VSLAM drift by ~1 deg.
AXIS_CANDIDATES = {
    "cv":       np.eye(3),                                          # OpenCV: X right, Y down, Z out
    "xy_flip":  np.diag([1.0, -1.0, -1.0]),                          # X right, Y up,  Z in
    "ros":      np.array([[0., -1., 0.], [0., 0., -1.], [1., 0., 0.]]),  # X out, Y left, Z up
    "ros_180":  np.array([[0., 1., 0.], [0., 0., -1.], [-1., 0., 0.]]),  # ros, yawed 180
}
BOARD_AXES = "cv"   # overwritten by the auto-select cell

def make_board(spec):
    d = cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, spec["dictionary"]))
    sx, sy = spec["squares"]; sq = spec["square_len"]; mk = sq * MARKER_RATIO
    try:                                  # OpenCV >= 4.7
        b = cv2.aruco.CharucoBoard((sx, sy), sq, mk, d)
        b.setLegacyPattern(True)
    except AttributeError:                # OpenCV 4.6
        b = cv2.aruco.CharucoBoard_create(sx, sy, sq, mk, d)
    return b, d

def board_object_points(spec, axes=None):
    """Interior ChArUco corners, (sx-1)*(sy-1) x 3, in the SURVEYED board frame."""
    sx, sy = spec["squares"]; sq = spec["square_len"]
    j, i = np.meshgrid(np.arange(1, sy), np.arange(1, sx), indexing="ij")
    P = np.column_stack([i.ravel() * sq, j.ravel() * sq, np.zeros(i.size)])
    P -= np.array([sx * sq / 2, sy * sq / 2, 0.0])           # board_origin = "center"
    Rc = AXIS_CANDIDATES[axes or BOARD_AXES]
    return P @ np.linalg.inv(Rc).T                           # p_board = Rc^-1 p_cv

_DET = {}
def detect_charuco(gray, spec):
    """-> (corner_ids (n,), uv (n,2)) in the board's ChArUco corner indexing."""
    key = (spec["dictionary"], spec["squares"], spec["square_len"])
    if key not in _DET: _DET[key] = make_board(spec)
    board, dic = _DET[key]
    if hasattr(cv2.aruco, "CharucoDetector"):
        det = cv2.aruco.CharucoDetector(board)
        cc, ci, _, _ = det.detectBoard(gray)
        if cc is None or len(cc) < 4: return None, None
        return ci.ravel().astype(int), cc.reshape(-1, 2)
    mc, mi, _ = cv2.aruco.detectMarkers(gray, dic)
    if mi is None or len(mi) < 2: return None, None
    n, cc, ci = cv2.aruco.interpolateCornersCharuco(mc, mi, gray, board)
    if n is None or n < 4: return None, None
    return ci.ravel().astype(int), cc.reshape(-1, 2)

def pnp_board(ids, uv, spec, K, axes=None):
    """-> (T_cam_board, mean_reproj_px) or (None, inf)."""
    P = board_object_points(spec, axes)[ids]
    if len(P) < 4: return None, np.inf
    ok, rv, tv = cv2.solvePnP(P.astype(np.float64), uv.astype(np.float64), K, None,
                              flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok: return None, np.inf
    rv, tv = cv2.solvePnPRefineLM(P.astype(np.float64), uv.astype(np.float64), K, None, rv, tv)
    proj, _ = cv2.projectPoints(P, rv, tv, K, None)
    err = float(np.mean(np.linalg.norm(proj.reshape(-1, 2) - uv, axis=1)))
    return Rt(cv2.Rodrigues(rv)[0], tv.ravel()), err

print("charuco helpers ready; axis candidates:", list(AXIS_CANDIDATES))

In [ ]:
# ============================== STEP 0b: run the census ==============================
FORCE = False
CENSUS = WORK / "census_m2.npz"
SPECS  = {n: BOARDS[n] for n in BOARDS}

if FORCE or not CENSUS.exists():
    # one detector pass per distinct (dictionary, geometry); collisions resolved later by position
    uniq, seen = [], set()
    for n, b in SPECS.items():
        k = (b["dictionary"], b["squares"], b["square_len"])
        if k not in seen: seen.add(k); uniq.append((k, b))
    print("detector passes:", [k for k, _ in uniq])

    rows, t0 = [], time.time()
    for fi, (t, m) in enumerate(iter_topic(BAG, T_INFRA1, stride=CENSUS_STRIDE, limit=LIMIT_FRAMES)):
        im = img_to_np(m)
        gray = im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        for k, spec in uniq:
            ids, uv = detect_charuco(gray, spec)
            if ids is None or len(ids) < MIN_CORNERS: continue
            T_cb, err = pnp_board(ids, uv, spec, K_IR)
            if T_cb is None or err > MAX_REPROJ_PX: continue
            rows.append(dict(t=t, frame=fi, key=k[0] + str(k[1]), n=len(ids),
                             rng=float(np.linalg.norm(T_cb[:3, 3])), err=err,
                             T=T_cb, ids=ids, uv=uv))
        if fi % 200 == 0:
            print(f"  {fi:5d} frames  {len(rows):4d} sightings  {time.time()-t0:5.1f}s", flush=True)
    np.savez_compressed(CENSUS, rows=np.array(rows, dtype=object), allow_pickle=True)
    print(f"detected {len(rows)} raw sightings in {time.time()-t0:.1f}s")

RAW = list(np.load(CENSUS, allow_pickle=True)["rows"])
print(f"loaded {len(RAW)} raw sightings")
if RAW:
    import collections
    print(collections.Counter(r["key"] for r in RAW))
    print(f"range {min(r['rng'] for r in RAW):.2f}-{max(r['rng'] for r in RAW):.2f} m, "
          f"mean reproj {np.mean([r['err'] for r in RAW]):.3f} px "
          f"(survey was 0.22-0.39 px; much worse => IR projector on, try USE_COLOR)")
else:
    print("!! NO SIGHTINGS. Check topic, MIN_CORNERS, and whether mobile_2 ever faced a board.")

In [ ]:
# ============================== load VSLAM odometry ==============================
vo_t, vo_T = [], []
for t, m in iter_topic(BAG, T_VO, limit=LIMIT_FRAMES):
    vo_t.append(t); vo_T.append(odom_to_T(m))
vo_t = np.array(vo_t); vo_T = np.array(vo_T)
print(f"VSLAM odometry: {len(vo_t)} poses, {vo_t[-1]-vo_t[0]:.1f} s, "
      f"path {np.sum(np.linalg.norm(np.diff(vo_T[:,:3,3],axis=0),axis=1)):.1f} m")

In [ ]:
# ============================== resolve the ID collision, cluster sightings ==============
# anchor and anchor_b are the same physical design, so a detection only tells us "some
# DICT_4X4_50 9x7 board". Sightings are clustered by board-origin position in the VSLAM
# frame (origins are invariant to the axis convention, so this runs before the axis check), then clusters are matched to surveyed boards by their distance to an unambiguous
# board. The surveyed rs->anchor (16.59 m) and rs->anchor_b (7.10 m) differ by 9.5 m, so the
# assignment survives metres of VSLAM drift.
SIGHT = []
for r in RAW:
    spec = next(b for b in SPECS.values() if b["dictionary"] + str(b["squares"]) == r["key"])
    T_cb, err = pnp_board(r["ids"], r["uv"], spec, K_IR, axes=BOARD_AXES)
    if T_cb is None or err > MAX_REPROJ_PX: continue
    T_vo = interp_traj(vo_t, vo_T, np.array([r["t"]]))[0]
    SIGHT.append(dict(**{k: r[k] for k in ("t", "n", "rng", "err", "ids", "uv", "key")},
                      T_cb=T_cb, p_vo=(T_vo @ T_cb)[:3, 3]))

def cluster(pts, tol=0.6):
    lab = -np.ones(len(pts), int); c = 0
    for i in range(len(pts)):
        if lab[i] >= 0: continue
        m = np.linalg.norm(pts - pts[i], axis=1) < tol
        lab[m] = c; c += 1
    return lab, c

CLUS = {}
for key in sorted({s["key"] for s in SIGHT}):
    idx = [i for i, s in enumerate(SIGHT) if s["key"] == key]
    P = np.array([SIGHT[i]["p_vo"] for i in idx])
    lab, nc = cluster(P)
    for c in range(nc):
        sel = [idx[k] for k in np.where(lab == c)[0]]
        CLUS[f"{key}#{c}"] = dict(key=key, idx=sel,
                                  p_vo=np.median([SIGHT[i]["p_vo"] for i in sel], axis=0),
                                  t0=min(SIGHT[i]["t"] for i in sel),
                                  t1=max(SIGHT[i]["t"] for i in sel))
print(f"{len(SIGHT)} sightings -> {len(CLUS)} spatial clusters")
for cid, c in CLUS.items():
    print(f"  {cid:28s} n={len(c['idx']):4d}  t=[{c['t0']-vo_t[0]:6.1f},{c['t1']-vo_t[0]:6.1f}]s"
          f"  p_vo={np.round(c['p_vo'],2)}")

# --- assign clusters to surveyed boards --------------------------------
unamb = {n: b for n, b in BOARDS.items()
         if not any(n in v for v in AMBIG.values())}
ref_cid = next((cid for cid, c in CLUS.items()
                if any(c["key"] == b["dictionary"] + str(b["squares"]) for b in unamb.values())), None)
ASSIGN = {}
if ref_cid is None:
    print("\n!! no unambiguous board seen - assignment cannot be resolved automatically.")
    print("   Set ASSIGN by hand, e.g. ASSIGN = {'DICT_4X4_50(9, 7)#0': 'anchor', ...}")
else:
    ref_name = next(n for n, b in unamb.items()
                    if b["dictionary"] + str(b["squares"]) == CLUS[ref_cid]["key"])
    ASSIGN[ref_cid] = ref_name
    p_ref = CLUS[ref_cid]["p_vo"]
    print(f"\nreference cluster {ref_cid} = '{ref_name}'")
    for cid, c in CLUS.items():
        if cid == ref_cid: continue
        d_meas = float(np.linalg.norm(c["p_vo"] - p_ref))
        cands = {n: float(np.linalg.norm(b["T_map_board"][:3, 3] -
                                         BOARDS[ref_name]["T_map_board"][:3, 3]))
                 for n, b in BOARDS.items()
                 if n != ref_name and b["dictionary"] + str(b["squares"]) == c["key"]}
        if not cands: continue
        best = min(cands, key=lambda n: abs(cands[n] - d_meas))
        ASSIGN[cid] = best
        others = " ".join(f"{n}={cands[n]:.2f}" for n in cands)
        print(f"  {cid:28s} d_meas={d_meas:6.2f} m -> '{best}'   (surveyed: {others})")
        if len(cands) > 1:
            gap = sorted(abs(cands[n] - d_meas) for n in cands)
            print(f"     margin {gap[1]-gap[0]:.2f} m  {'OK' if gap[1]-gap[0] > 1.0 else '<< AMBIGUOUS'}")

for cid, name in ASSIGN.items():
    for i in CLUS[cid]["idx"]: SIGHT[i]["board"] = name
SIGHT = [s for s in SIGHT if "board" in s]
print(f"\n{len(SIGHT)} assigned sightings across {len(set(s['board'] for s in SIGHT))} boards")

In [ ]:
# ============================== board-axis convention check ==============================
# PnP under ANY rigid axis convention reproduces the same corners and the same board origin -
# the convention only redefines the board's ORIENTATION frame. So it cannot be picked from
# mobile_2's detections alone; it must match what the SURVEY used when it stored each qxyzw.
# The discriminating signal: the relative rotation between two boards, survey-stored vs
# measured through VSLAM. A wrong convention is off by ~90-180 deg; VSLAM rotation drift over
# the between-board stretch is ~1 deg. Needs sightings of >= 2 distinct boards.
def rel_rot_score(axes):
    Rm = {}
    for s in SIGHT:
        T_cb, err = pnp_board(s["ids"], s["uv"], BOARDS[s["board"]], K_IR, axes=axes)
        if T_cb is None or err > MAX_REPROJ_PX: continue
        T_vo = interp_traj(vo_t, vo_T, np.array([s["t"]]))[0]
        Rm.setdefault(s["board"], []).append((T_vo @ T_cb)[:3, :3])
    names = sorted(Rm)
    if len(names) < 2: return None
    worst = 0.0
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            R_meas = Rm[a][len(Rm[a]) // 2].T @ Rm[b][len(Rm[b]) // 2]
            R_srv = BOARDS[a]["T_map_board"][:3, :3].T @ BOARDS[b]["T_map_board"][:3, :3]
            worst = max(worst, math.degrees(np.linalg.norm(
                Rot.from_matrix(R_srv.T @ R_meas).as_rotvec())))
    return worst                      # deg; the right convention is the ~drift-sized one

scores = {a: rel_rot_score(a) for a in AXIS_CANDIDATES}
if all(v is None for v in scores.values()):
    print("!! fewer than 2 distinct boards sighted - the convention cannot be checked from "
          "this bag.\n   Set BOARD_AXES by hand from the survey tool's own definition "
          "(its config says board_axes='ros').")
else:
    for a, v in sorted(scores.items(), key=lambda kv: (kv[1] is None, kv[1])):
        print(f"  {a:9s} relative-rotation disagreement "
              + ("   n/a" if v is None else f"{v:8.2f} deg"))
    BOARD_AXES = min((a for a in scores if scores[a] is not None), key=lambda a: scores[a])
    best = scores[BOARD_AXES]
    print(f"\n-> BOARD_AXES = '{BOARD_AXES}'  ({best:.2f} deg)")
    if best > 10:
        print("!! even the best candidate disagrees by >10 deg - none of the enumerated "
              "conventions matches the survey. Read the survey tool's board-frame code and "
              "add its rotation to AXIS_CANDIDATES.")
    # re-solve every stored sighting pose under the chosen convention
    for s in SIGHT:
        T_cb, err = pnp_board(s["ids"], s["uv"], BOARDS[s["board"]], K_IR, axes=BOARD_AXES)
        if T_cb is not None: s["T_cb"], s["err"] = T_cb, err

In [ ]:
# ============================== CENSUS GATE ==============================
t_rel = np.array([s["t"] for s in SIGHT]) - vo_t[0]
bnames = sorted({s["board"] for s in SIGHT})
dur = vo_t[-1] - vo_t[0]

# a "window" = contiguous sightings of one board with < 2 s gaps
WINDOWS = []
for b in bnames:
    tb = np.sort(t_rel[[i for i, s in enumerate(SIGHT) if s["board"] == b]])
    if not len(tb): continue
    br = np.where(np.diff(tb) > 2.0)[0]
    for a, z in zip(np.r_[0, br + 1], np.r_[br, len(tb) - 1]):
        WINDOWS.append((b, tb[a], tb[z], z - a + 1))

print(f"trajectory {dur:.1f} s   sighting duty cycle "
      f"{100*len(SIGHT)/max(1,len(vo_t)/CENSUS_STRIDE):.1f}%")
print(f"\n{'board':11s} {'t_start':>8s} {'t_end':>8s} {'n':>5s}")
for b, a, z, n in sorted(WINDOWS, key=lambda w: w[1]):
    print(f"{b:11s} {a:8.1f} {z:8.1f} {n:5d}")
gaps = np.diff(np.r_[0, sorted(w[1] for w in WINDOWS), dur])
print(f"\n{len(WINDOWS)} windows; longest board-free stretch {gaps.max():.1f} s "
      f"({gaps.max()*0.8:.1f} m at 0.8 m/s)")
print("GATE:", "PASS - three-arm ablation is meaningful" if len(WINDOWS) >= 4 else
      "FAIL - reframe as anchored vs unanchored VSLAM, not a fusion ablation")

fig, ax = plt.subplots(2, 1, figsize=(11, 6), height_ratios=[1, 2])
for i, b in enumerate(bnames):
    m = [j for j, s in enumerate(SIGHT) if s["board"] == b]
    ax[0].scatter(t_rel[m], np.full(len(m), i), s=8, label=b)
    ax[1].scatter(t_rel[m], [SIGHT[j]["rng"] for j in m], s=8, label=b)
ax[0].set_yticks(range(len(bnames))); ax[0].set_yticklabels(bnames)
ax[0].set_xlim(0, dur); ax[0].set_title("board sightings over the trajectory"); ax[0].grid(alpha=.3)
ax[1].set_xlim(0, dur); ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("range [m]")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Step 0c - are the boards still where the survey says?  **(GATE)**

The survey is stamped `1787894041`; this bag starts `1787899802` - **96 minutes later**. The
surveyed board poses only transfer if nothing was bumped in between.

The check below is extrinsic-free: measure each inter-board baseline *inside this bag* by
composing two sightings through the VSLAM trajectory, and compare to the survey. It is a
**joint** test of board stability and VSLAM metric scale, so read it that way - agreement
validates both; disagreement tells you something is wrong without saying which. That is still
worth having before you build three estimators on top of these boards.

The stronger, single-hypothesis test is to redetect the boards from `mobile_1`'s ZED in this
same bag, since `mobile_1` carries the Ouster and so has a GLIM pose in the map frame. It needs
the LiDAR->ZED extrinsic, which is not in this bag's 3-message `/tf_static`; do it separately
if this cell comes back disagreeing.

In [ ]:
# ============================== STEP 0c: board stability / VO scale ==============
print(f"{'pair':26s} {'surveyed':>9s} {'measured':>9s} {'diff':>8s} {'n':>5s}")
rows = []
for i in range(len(bnames)):
    for j in range(i + 1, len(bnames)):
        a, b = bnames[i], bnames[j]
        d_srv = float(np.linalg.norm(BOARDS[a]["T_map_board"][:3, 3] -
                                     BOARDS[b]["T_map_board"][:3, 3]))
        pa = np.array([s["p_vo"] for s in SIGHT if s["board"] == a])
        pb = np.array([s["p_vo"] for s in SIGHT if s["board"] == b])
        d_msr = float(np.linalg.norm(np.median(pa, 0) - np.median(pb, 0)))
        rows.append((d_srv, d_msr))
        print(f"{a+' <-> '+b:26s} {d_srv:8.3f}m {d_msr:8.3f}m {1000*(d_msr-d_srv):+7.0f}mm"
              f" {len(pa)+len(pb):5d}")
if len(rows) >= 2:
    s = np.polyfit([r[0] for r in rows], [r[1] for r in rows], 1)[0]
    print(f"\nimplied VSLAM scale {s:.5f}  ({(s-1)*1e6:+.0f} ppm)")
    print("  interpret: a consistent scale != 1 across pairs => VSLAM stereo scale error "
          "(estimable, ESTIMATE_VO_SCALE=True handles it).")
    print("  a single pair disagreeing while others match => that board moved.")
elif rows:
    print("\nonly one pair - cannot separate 'board moved' from 'VSLAM scale'. "
          "Treat a large diff as a warning, not a diagnosis.")

## Step 1 - depth -> point cloud

Deprojection plus three filters that matter more than they look:

- **range gate 0.4-3.5 m.** D455 stereo error grows as z^2 (~2% at 4 m ~ 8 cm, thicker than the
  reference map). 3.5 m keeps depth noise under the map's 2.9 cm sensor floor.
- **edge reject.** Stereo emits flying pixels at every occlusion boundary; unfiltered they are
  the single largest source of ICP bias here, because they always lie *between* two surfaces.
- **voxel centroid, never voxel centre.** Voxel centres cost 2.4 cm of quantisation in the
  LiDAR pipeline; the same bug would be invisible-but-fatal here.

No deskew: infra1 and depth are global shutter, unlike the Ouster.

In [ ]:
# ============================== STEP 1: depth -> cloud ==============================
def voxel_centroid(P, v):
    """Average of the points in each voxel (NOT the voxel centre)."""
    q = np.floor(P / v).astype(np.int64)
    q -= q.min(0)
    key = (q[:, 0] << 40) | (q[:, 1] << 20) | q[:, 2]
    order = np.argsort(key, kind="stable"); key = key[order]; Ps = P[order]
    br = np.r_[0, np.flatnonzero(np.diff(key)) + 1, len(key)]
    cs = np.vstack([np.zeros(3), np.cumsum(Ps, 0)])
    return (cs[br[1:]] - cs[br[:-1]]) / np.diff(br)[:, None]

def depth_to_cloud(d16, K):
    z = d16.astype(np.float32) * DEPTH_SCALE
    z[(z < RANGE_MIN) | (z > RANGE_MAX)] = 0
    zz = np.where(z > 0, z, np.nan)
    hi = ndimage.maximum_filter(np.nan_to_num(zz, nan=-1e3), size=3)
    lo = ndimage.minimum_filter(np.nan_to_num(zz, nan= 1e3), size=3)
    z[(hi - lo) > EDGE_JUMP] = 0                      # flying pixels at occlusion boundaries
    v, u = np.nonzero(z)
    if len(u) < 100: return np.zeros((0, 3), np.float32)
    zc = z[v, u]
    P = np.column_stack([(u - K[0, 2]) * zc / K[0, 0],
                         (v - K[1, 2]) * zc / K[1, 1], zc]).astype(np.float32)
    P = voxel_centroid(P, CLOUD_VOXEL)
    if len(P) > MAX_PTS_PER_FR:
        P = P[np.random.default_rng(0).choice(len(P), MAX_PTS_PER_FR, replace=False)]
    return P.astype(np.float32)
print("cloud helpers ready")

In [ ]:
# ---- extract ----
FORCE = False
CLOUDS = WORK / "clouds.npz"
K_D, _, W_D, H_D = camera_K(BAG, T_DEPTH_CI)
if FORCE or not CLOUDS.exists():
    keep_dt = 1.0 / REG_RATE_HZ
    ts, blobs, t_last = [], [], -1e9
    t0 = time.time()
    for i, (t, m) in enumerate(iter_topic(BAG, T_DEPTH, limit=LIMIT_FRAMES)):
        if t - t_last < keep_dt: continue
        P = depth_to_cloud(img_to_np(m), K_D)
        if len(P) < 500: continue
        t_last = t; ts.append(t); blobs.append(P)
        if len(ts) % 100 == 0:
            print(f"  {len(ts):5d} clouds  mean {np.mean([len(b) for b in blobs]):6.0f} pts"
                  f"  {time.time()-t0:5.1f}s", flush=True)
    np.savez_compressed(CLOUDS, ts=np.array(ts),
                        sizes=np.array([len(b) for b in blobs]),
                        pts=np.concatenate(blobs).astype(np.float32))
    print(f"{len(ts)} clouds in {time.time()-t0:.1f}s")

_z = np.load(CLOUDS)
CL_T = _z["ts"]; _sz = _z["sizes"]; _off = np.r_[0, np.cumsum(_sz)]; _pts = _z["pts"]
CL = [_pts[_off[i]:_off[i + 1]] for i in range(len(CL_T))]
print(f"{len(CL)} clouds @ {len(CL)/(CL_T[-1]-CL_T[0]):.1f} Hz, "
      f"{_sz.mean():.0f} pts mean ({_sz.min()}-{_sz.max()}), "
      f"median range {np.median(np.linalg.norm(CL[len(CL)//2],axis=1)):.2f} m")

## Step 2 - the frozen reference map

Same target construction as `01a_refine_poses.py`: local planes on `PLANE_VOXEL` patches with a
**soft** planarity weight (`w = 1 - min(ratio/planarity, 1)`), matched to query points by voxel
membership rather than nearest centroid.

Both details were paid for the hard way. A hard planarity gate is what makes HBA reject exactly
the smeared surfaces you need it to fix - measured on this data, a 0.35 gate admitted 52% of wall
cells at 7.6 cm median thickness while rejecting cells at 15.3 cm. And matching planes by nearest
centroid drags floor points onto wall planes, producing 57 cm "corrections".

In [ ]:
# ============================== STEP 2: reference map ==============================
def read_ply_xyz(path):
    try:
        import open3d as o3d
        return np.asarray(o3d.io.read_point_cloud(str(path)).points)
    except ImportError:
        pass
    with open(path, "rb") as f:                      # minimal binary/ascii PLY fallback
        hdr, n, fmt, props = [], 0, None, []
        while True:
            l = f.readline().decode("ascii", "ignore").strip(); hdr.append(l)
            if l.startswith("format"): fmt = l.split()[1]
            if l.startswith("element vertex"): n = int(l.split()[2])
            if l.startswith("property"): props.append(l.split()[-1])
            if l == "end_header": break
        if fmt == "ascii":
            a = np.loadtxt(f, max_rows=n)
        else:
            a = np.frombuffer(f.read(n * 4 * len(props)), np.float32).reshape(n, len(props))
        return np.asarray(a[:, [props.index(c) for c in "xyz"]], float)

class Reference:
    """KD-tree + per-voxel local planes over the frozen map."""
    def __init__(self, P, voxel=TARGET_VOXEL, plane_voxel=PLANE_VOXEL,
                 k=NORMAL_K, planarity=1.0, min_pts=12):
        self.P = voxel_centroid(np.asarray(P, float), voxel)
        self.tree = cKDTree(self.P)
        self.pv = plane_voxel
        self.origin = self.P.min(0)
        q = self._vox(self.P)
        key = (q[:, 0] << 40) | (q[:, 1] << 20) | q[:, 2]
        o = np.argsort(key, kind="stable"); key_s = key[o]; Ps = self.P[o]
        br = np.r_[0, np.flatnonzero(np.diff(key_s)) + 1, len(key_s)]
        self.C, self.N, self.W, ks = [], [], [], []
        for a, b in zip(br[:-1], br[1:]):
            if b - a < min_pts: continue
            X = Ps[a:b]; c = X.mean(0)
            ev, V = np.linalg.eigh((X - c).T @ (X - c) / (b - a))
            ratio = ev[0] / max(ev[1], 1e-12)
            self.C.append(c); self.N.append(V[:, 0]); ks.append(key_s[a])
            self.W.append(1.0 - min(ratio / planarity, 1.0))     # SOFT, not a gate
        self.C = np.array(self.C); self.N = np.array(self.N); self.W = np.array(self.W)
        self.idx = {int(k): i for i, k in enumerate(ks)}
        print(f"reference: {len(self.P)} pts, {len(self.C)} plane cells, "
              f"mean planarity weight {self.W.mean():.3f}")

    def _vox(self, P):
        q = np.floor((P - self.origin) / self.pv).astype(np.int64)
        return np.clip(q, 0, (1 << 20) - 1)

    def plane_of(self, Q):
        """Plane target by VOXEL MEMBERSHIP (never nearest centroid). -> c, n, w, mask"""
        q = self._vox(Q); key = (q[:, 0] << 40) | (q[:, 1] << 20) | q[:, 2]
        j = np.array([self.idx.get(int(k), -1) for k in key])
        m = j >= 0
        return self.C[j[m]], self.N[j[m]], self.W[j[m]], m

print("Reference class ready")

In [ ]:
REF = Reference(read_ply_xyz(REF_MAP))

## Step 2b - board-free initial alignment

Arm A must not be seeded from the boards, or the ablation leaks: an arm that "uses no fiducials"
but starts from a fiducial-derived pose is not board-free. So the seed comes from a global
registration of an accumulated depth cloud (FPFH + RANSAC, then point-to-plane ICP) - the same
coarse-alignment machinery as `cross_session.py`.

The board-derived Umeyama fit is computed too, but only as a **cross-check**: if the two seeds
disagree by more than a few centimetres, one of them is wrong and it is worth knowing before
three estimators are built on top.

In [ ]:
# ============================== STEP 2b: seed T_map_vo ==============================
import open3d as o3d
VOXEL_COARSE = 0.25

def _o3d(P, v):
    p = o3d.geometry.PointCloud(); p.points = o3d.utility.Vector3dVector(np.asarray(P, float))
    p = p.voxel_down_sample(v)
    p.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=v * 3, max_nn=30))
    return p

def global_align(src, dst, v=VOXEL_COARSE):
    a, b = _o3d(src, v), _o3d(dst, v)
    fa = o3d.pipelines.registration.compute_fpfh_feature(
        a, o3d.geometry.KDTreeSearchParamHybrid(radius=v * 5, max_nn=100))
    fb = o3d.pipelines.registration.compute_fpfh_feature(
        b, o3d.geometry.KDTreeSearchParamHybrid(radius=v * 5, max_nn=100))
    r = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        a, b, fa, fb, True, v * 1.5,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(False), 3,
        [o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
         o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(v * 1.5)],
        o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 0.999))
    fine = o3d.pipelines.registration.registration_icp(
        a, b, v * 0.6, r.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane())
    return np.array(fine.transformation), fine.fitness, fine.inlier_rmse

# accumulate the first SEED_SECONDS of depth clouds in the VSLAM frame
SEED_SECONDS = 20.0
sel = np.where(CL_T - CL_T[0] < SEED_SECONDS)[0]
T_vo_i = interp_traj(vo_t, vo_T, CL_T[sel])
acc = np.vstack([apply(T_vo_i[k], CL[i]) for k, i in enumerate(sel)])
acc = voxel_centroid(acc, 0.05)
print(f"seed cloud: {len(sel)} frames, {len(acc)} pts")

T_map_vo, fit, rmse = global_align(acc, REF.P)
print(f"global seed  fitness {fit:.3f}  inlier rmse {rmse*100:.2f} cm")
print(np.round(T_map_vo, 4))
if fit < 0.5:
    print("!! low fitness - the accumulated frustum may not be distinctive enough. "
          "Raise SEED_SECONDS, or seed from the boards and NOTE the leak in the paper.")

# cross-check against a board-derived fit (NOT used to seed arm A)
if len(bnames) >= 3:
    Pv = np.array([np.median([s["p_vo"] for s in SIGHT if s["board"] == b], 0) for b in bnames])
    Pm = np.array([BOARDS[b]["T_map_board"][:3, 3] for b in bnames])
    cv_, cm_ = Pv.mean(0), Pm.mean(0)
    U, _, Vt = np.linalg.svd((Pm - cm_).T @ (Pv - cv_))
    D = np.diag([1, 1, np.sign(np.linalg.det(U @ Vt))])
    T_board = Rt(U @ D @ Vt, cm_ - (U @ D @ Vt) @ cv_)
    d = np.linalg.norm(T_board[:3, 3] - T_map_vo[:3, 3])
    a = math.degrees(np.linalg.norm(log_R(T_board[:3, :3].T @ T_map_vo[:3, :3])))
    print(f"\nboard-derived seed disagrees by {d*100:.1f} cm / {a:.2f} deg "
          f"({'consistent' if d < 0.10 else 'CHECK - one of them is wrong'})")
else:
    print("\n<3 boards seen: no board cross-check possible")

## Arm A, part 1 - the chained scan-to-map pass

This produces the **initialisation** for every arm and the per-frame conditioning diagnostics.
It is not itself arm A: arm A is this geometry re-expressed as graph factors, in the arms cell
below. Keeping the chained poses out of the graph is deliberate and is explained there.

The central difficulty, stated plainly: the Ouster sees 360 deg, the D455 sees an **87 deg
frustum**. Facing a flat wall, a frustum constrains *one* direction - the wall normal - and
nothing else. In these corridors that is not an edge case, it is most frames.

So per-frame degeneracy handling is mandatory. After each ICP the 6x6 Hessian is
eigen-decomposed (in a length-normalised basis, so translation and rotation eigenvalues are
comparable) and directions below `DEGEN_RATIO` of the largest are marked unobservable. Those
directions are:

- **damped in the solve** (Tikhonov, so the frame does not slide along the corridor), and
- **reported**, as the `nobs` diagnostic.

They are *not* thresholded away. Measured spectra here are graded, not bimodal - mid-corridor
this synthetic case gives `[0.003 0.004 0.015 0.040 0.55 1.0]`, with no gap to cut at. The full
Hessian is handed to the pose graph as the factor information, which is precisely what an
information matrix is for: a direction the frustum cannot see arrives with a small eigenvalue
and the graph down-weights it on its own. Hard-thresholding measurably *hurt* here - it discards
the 0.015 and 0.040 directions, which carry real constraint, and lets VSLAM re-drift into the
hole. Rank reduction is only correct when a direction is exactly unobservable.

This also handles the correlation between consecutive ICP factors gracefully. Each frame's ICP is
seeded from the previous one, so along the corridor axis `A_T[i]` mostly carries the seed rather
than new information - and the small eigenvalue in that direction says exactly that.

In [ ]:
# ============================== chained scan-to-map pass (init + diagnostics) ==============================
def icp_frame(P_body, T_init, ref, L=None):
    """Point-to-plane ICP of one depth cloud against the frozen map.
    Returns T, info (6x6 Hessian, [dt_world, dphi_body] convention), n_used, rms,
    eig (length-normalised spectrum, diagnostic), n_obs (count above DEGEN_RATIO,
    diagnostic only -- never used to discard information)."""
    T = T_init.copy(); info = np.zeros((6, 6)); nu = 0; rms = np.nan
    eig = np.zeros(6); nobs = 0
    if L is None:            # express rotation in metres of point displacement
        L = 1.0 / max(float(np.median(np.linalg.norm(P_body, axis=1))), 1e-3)
    S = np.diag([1, 1, 1, L, L, L])
    for gate in MAX_CORR:
        for _ in range(ITERS_PER_GATE):
            Q = apply(T, P_body)
            c, n, w, m = ref.plane_of(Q)
            if m.sum() < 50: break
            p = P_body[m]; R = T[:3, :3]
            r = np.einsum("ij,ij->i", Q[m] - c, n)
            keep = np.abs(r) < gate
            if keep.sum() < 50: break
            c, n, w, p, r = c[keep], n[keep], w[keep], p[keep], r[keep]
            hub = np.minimum(1.0, HUBER / np.maximum(np.abs(r), 1e-9))
            ww = w * hub
            Rn = n @ R                                    # (N,3) == (R^T n)^T
            J = np.hstack([n, np.cross(p, Rn)])           # dr/d[dt_world, dphi_body]
            Jw = J * ww[:, None]
            H = J.T @ Jw; g = Jw.T @ r
            damp = PRIOR_BETA * np.trace(H) / 6.0
            d = -np.linalg.solve(H + damp * np.eye(6), g)
            T = Rt(R @ exp_r(d[3:]), T[:3, 3] + d[:3])
            nu = int(keep.sum()); rms = float(np.sqrt(np.mean(ww * r * r) / max(ww.mean(), 1e-9)))
            # The information matrix IS the degeneracy handling: a direction the frustum
            # cannot see comes out with a small eigenvalue and the graph weights it down
            # accordingly. Do NOT threshold it -- measured spectra here are graded
            # ([0.003 0.004 0.015 0.040 0.55 1.0] mid-corridor), so a hard cut throws away
            # real constraint. eig/nobs below are REPORTING ONLY.
            info = H
            ev = np.linalg.eigvalsh(S @ H @ S)            # length-normalised, comparable units
            eig = ev / max(ev.max(), 1e-12)
            nobs = int((eig > DEGEN_RATIO).sum())
            if np.linalg.norm(d[:3]) < 1e-4 and np.linalg.norm(d[3:]) < 1e-5: break
    return T, info, nu, rms, eig, nobs
print("icp_frame ready")

In [ ]:
# ---- run the chained pass ----
FORCE = False
ARM_A = WORK / "arm_a.npz"
if FORCE or not ARM_A.exists():
    T_vo_all = interp_traj(vo_t, vo_T, CL_T)
    Ts = np.zeros((len(CL), 4, 4)); INFO = np.zeros((len(CL), 6, 6))
    NU = np.zeros(len(CL), int); RMS = np.zeros(len(CL)); NOBS = np.zeros(len(CL), int)
    EIG = np.zeros((len(CL), 6))
    T_prev = T_map_vo @ T_vo_all[0]; t0 = time.time()
    for i in range(len(CL)):
        # seed: previous solved pose advanced by the VSLAM relative motion (never the boards)
        if i: T_prev = T_prev @ (inv(T_vo_all[i - 1]) @ T_vo_all[i])
        Ts[i], INFO[i], NU[i], RMS[i], EIG[i], NOBS[i] = icp_frame(CL[i], T_prev, REF)
        T_prev = Ts[i]
        if i % 100 == 0:
            print(f"  {i:5d}/{len(CL)}  used {NU[i]:5d}  rms {RMS[i]*100:5.2f} cm  "
                  f"obs {NOBS[i]}/6  {time.time()-t0:6.1f}s", flush=True)
    np.savez_compressed(ARM_A, T=Ts, info=INFO, nu=NU, rms=RMS, nobs=NOBS, eig=EIG, ts=CL_T)
    print(f"arm A done in {time.time()-t0:.1f}s")

_a = np.load(ARM_A)
A_T, A_INFO, A_NOBS, A_RMS, A_EIG = _a["T"], _a["info"], _a["nobs"], _a["rms"], _a["eig"]
import collections
print("observable DOF histogram:", dict(sorted(collections.Counter(A_NOBS.tolist()).items())))
print(f"fully observable (6/6): {100*np.mean(A_NOBS==6):.1f}% of frames")
print(f"rank-deficient  (<6):   {100*np.mean(A_NOBS<6):.1f}%   <- this is why boards matter")
print(f"weakest direction carries {np.median(A_EIG[:,0]):.4f} of the strongest (median frame)")
print(f"plane rms: median {np.nanmedian(A_RMS)*100:.2f} cm, p95 {np.nanpercentile(A_RMS,95)*100:.2f} cm")
write_traj(WORK / "traj_m2_chained_icp.txt", CL_T, A_T)   # initialisation, not an arm

## Step 3 - hand-eye: what frame is VSLAM actually in?

`visual_slam/tracking/odometry` is published in some frame rigidly attached to the robot, not
necessarily infra1 optical. Relative motion in the two frames differs by that extrinsic:
`Z_cam = X^-1 Z_V X`. This bag's `/tf_static` carries only 3 messages and may not contain the
RealSense tree, so `X` is estimated instead - Park-Martin `AX = XB` against arm A's trajectory,
using only frames arm A found fully observable.

If the residual comes back large, VSLAM relative motion is not trustworthy as a graph
constraint and arms B/C need `/tf` inspected by hand.

In [ ]:
# ============================== hand-eye AX = XB ==============================
def hand_eye(A_list, B_list):
    """Solve X in B X = X A (Park & Martin). A: motion in cam frame, B: in VSLAM frame."""
    a = np.array([log_R(A[:3, :3]) for A in A_list])
    b = np.array([log_R(B[:3, :3]) for B in B_list])
    U, _, Vt = np.linalg.svd(b.T @ a)
    Rx = U @ np.diag([1, 1, np.sign(np.linalg.det(U @ Vt))]) @ Vt
    M, r = [], []
    for A, B in zip(A_list, B_list):
        M.append(B[:3, :3] - np.eye(3)); r.append(Rx @ A[:3, 3] - B[:3, 3])
    tx = np.linalg.lstsq(np.vstack(M), np.concatenate(r), rcond=None)[0]
    return Rt(Rx, tx)

STEP = max(1, int(0.5 * REG_RATE_HZ))       # ~0.5 s motion increments
T_vo_all = interp_traj(vo_t, vo_T, CL_T)
good = np.where(A_NOBS == 6)[0]
pairs = [(i, i + STEP) for i in good if i + STEP < len(CL) and A_NOBS[i + STEP] == 6]
pairs = [(i, j) for i, j in pairs
         if np.linalg.norm(log_R((inv(A_T[i]) @ A_T[j])[:3, :3])) > 0.05]   # need rotation
print(f"hand-eye from {len(pairs)} motion pairs (of {len(good)} observable frames)")
if len(pairs) >= 20:
    Am = [inv(A_T[i]) @ A_T[j] for i, j in pairs]
    Bm = [inv(T_vo_all[i]) @ T_vo_all[j] for i, j in pairs]
    X = hand_eye(Am, Bm)
    res_t = [np.linalg.norm((inv(X) @ B @ X)[:3, 3] - A[:3, 3]) for A, B in zip(Am, Bm)]
    res_r = [math.degrees(np.linalg.norm(log_R(((inv(X) @ B @ X)[:3, :3]).T @ A[:3, :3])))
             for A, B in zip(Am, Bm)]
    print(f"X = T_V_cam  t={np.round(X[:3,3],4)}  "
          f"rpy={np.round(Rot.from_matrix(X[:3,:3]).as_euler('xyz',degrees=True),2)}")
    print(f"residual: {np.median(res_t)*1000:.1f} mm / {np.median(res_r):.2f} deg (median)")
    if np.median(res_t) > 0.05:
        print("!! large residual - VSLAM relative motion disagrees with arm A. "
              "Inspect /tf before trusting the odometry factors.")
else:
    X = np.eye(4)
    print("!! too few rotating, observable pairs - falling back to X = I. "
          "VSLAM factors will absorb the extrinsic as a systematic error.")

## Step 3b - measure the VSLAM noise, do not guess it

The single most sensitive knob in the graph. Set 5x too loose and the trajectory wanders between
scan-to-map constraints; set it too tight and VSLAM drift is forced onto the map. On the synthetic
corridor the difference between a guessed and a measured sigma was **2.47 cm vs 0.27 cm** of ATE,
against a per-frame ICP noise floor of 0.29 cm - i.e. the whole error budget was this one number.

So it is measured: compare VSLAM relative motion to arm A's relative motion over the frames arm A
found best conditioned. The result is an *upper bound* (it contains ICP's own noise), which is the
safe direction to err.

In [ ]:
# ============================== measure VSLAM relative noise ==============================
best = np.where(A_EIG[:, 0] > np.percentile(A_EIG[:, 0], 70))[0]     # best-conditioned frames
pair = [(i, i + 1) for i in best if i + 1 < len(CL) and (i + 1) in set(best.tolist())]
print(f"{len(pair)} consecutive well-conditioned pairs of {len(CL)} frames")
if len(pair) >= 30:
    dt_n = np.array([CL_T[j] - CL_T[i] for i, j in pair])
    et, er = [], []
    for (i, j), d in zip(pair, dt_n):
        Za = inv(A_T[i]) @ A_T[j]
        Zv = inv(X) @ inv(interp_traj(vo_t, vo_T, np.array([CL_T[i]]))[0]) @ \
             interp_traj(vo_t, vo_T, np.array([CL_T[j]]))[0] @ X
        k = 0.1 / max(d, 1e-3)                       # normalise to a 0.1 s interval
        et.append(np.linalg.norm(Za[:3, 3] - Zv[:3, 3]) * k)
        er.append(np.linalg.norm(log_R(Za[:3, :3].T @ Zv[:3, :3])) * k)
    VO_SIGMA_T = float(np.percentile(et, 68))
    VO_SIGMA_R = float(np.percentile(er, 68))
    print(f"measured per 0.1 s: sigma_t {VO_SIGMA_T*1000:.2f} mm   "
          f"sigma_R {math.degrees(VO_SIGMA_R)*60:.2f} arcmin")
    print("  (upper bound - it contains ICP's own per-frame noise, which is the safe direction)")
else:
    print(f"too few well-conditioned pairs; keeping the defaults "
          f"{VO_SIGMA_T*1000:.1f} mm / {math.degrees(VO_SIGMA_R)*60:.1f} arcmin")

## Arms B and C - one pose graph, factors switched on and off

All three arms are the **same estimator**; they differ only in which factor types are enabled.
Running three separate pipelines would make the comparison meaningless.

Nodes are placed at every registration stamp **and** at every board sighting stamp, so a sighting
is never snapped onto a node up to 50 ms away (at 0.8 m/s that alone would be 4 cm of error).

**Board constraints are per-corner reprojection factors, not composed 6-DoF pose priors.** The
boards are 18x14 cm (`anchor`, `anchor_b`) and 22x17 cm (`rs_anchor`). A planar target that small
at 1-2 m has weak out-of-plane rotation observability - which is exactly what the survey's
`max_deg` of 1.6-4.6 deg is reporting. A Gaussian pose prior fabricates rotation confidence the
geometry does not have; reprojection factors keep the real geometry and self-weight correctly.

The surveyed board's own uncertainty is folded into the pixel sigma: at range r a board position
error `sigma_t` projects to `f*sigma_t/r` pixels. For `anchor_b` at 1.5 m that is ~3.7 px against
0.5 px of corner noise - so board *quality* dominates board *detection*, and weighting all three
boards equally would be the dominant error.

**ICP enters as point-to-plane residuals re-linearised every iteration, not as a pose prior.**
This matters more than it sounds. The chained ICP pass produces poses that are correlated -
each frame is seeded from the one before - so injecting them as independent absolute
measurements lets the chain's own drift accumulate as evidence. Measured on the synthetic
corridor: along the unobservable axis that pseudo-evidence reached 2.3e6 of information against
a board's 1.9e6, and the boards lost. Residuals evaluated at the current estimate contain no
seed at all. The chained pass survives only as the initialisation and as the `nobs` diagnostic.

In [ ]:
# ============================== pose graph ==============================
ICP_SIGMA   = 0.02          # m; the reference map's own surface noise - ICP cannot beat it
ICP_PTS     = 400           # points per frame contributed to the graph
BOARD_HUBER = 3.0           # px
GAUGE_W     = 1e-2          # only prevents a singular system; must NOT pin the seed pose

def build_nodes():
    st = np.array(sorted({round(s["t"], 6) for s in SIGHT}))
    ts = np.unique(np.round(np.r_[CL_T, st], 6))
    icp_at = {round(t, 6): i for i, t in enumerate(np.round(CL_T, 6))}
    node_icp = np.array([icp_at.get(round(t, 6), -1) for t in ts])
    sg = {}
    for s in SIGHT:
        sg.setdefault(int(np.searchsorted(ts, round(s["t"], 6))), []).append(s)
    return ts, node_icp, sg

def board_corners_map(name, ids):
    b = BOARDS[name]
    return apply(b["T_map_board"], board_object_points(b, BOARD_AXES)[ids])

def project(K, Pc):
    z = np.maximum(Pc[:, 2], 1e-6)
    return np.column_stack([K[0, 0] * Pc[:, 0] / z + K[0, 2],
                            K[1, 1] * Pc[:, 1] / z + K[1, 2]])

def solve_graph(use_icp=True, use_board=True, boards_used=None, verbose=True):
    """Gauss-Newton over node poses (+ optional VSLAM scale). Returns (Ts, s, report)."""
    boards_used = set(boards_used if boards_used is not None else BOARDS)
    n = len(NODE_T); NS = 6 * n
    ns = 1 if ESTIMATE_VO_SCALE else 0
    # initial guess: arm A where available, else VSLAM chained from the seed
    T_vo_n = interp_traj(vo_t, vo_T, NODE_T)
    Ts = np.array([A_T[NODE_ICP[k]] if NODE_ICP[k] >= 0 else T_map_vo @ T_vo_n[k] @ X
                   for k in range(n)]) if use_icp else \
         np.array([T_map_vo @ T_vo_n[k] @ X for k in range(n)])
    s = 1.0
    # VSLAM relative measurements between consecutive nodes, in the cam frame
    Z = np.array([inv(X) @ inv(T_vo_n[k]) @ T_vo_n[k + 1] @ X for k in range(n - 1)])
    dt = np.diff(NODE_T)
    rep = {}
    for it in range(GN_ITERS):
        I_, J_, V_, r_ = [], [], [], []
        def add(rows, cols, vals, res):
            base = len(r_); r_.extend(res)
            I_.extend((np.asarray(rows) + base).tolist()); J_.extend(cols); V_.extend(vals)
        # ---- prior on node 0 (gauge) ----
        add(np.arange(6), list(range(6)), [GAUGE_W] * 6, list(np.zeros(6)))
        # ---- VSLAM relative ----
        for k in range(n - 1):
            Ti, Tj, Zm = Ts[k], Ts[k + 1], Z[k]
            Ri, Rj, Rm = Ti[:3, :3], Tj[:3, :3], Zm[:3, :3]
            d = Tj[:3, 3] - Ti[:3, 3]
            rt = Ri.T @ d - s * Zm[:3, 3]
            rr = log_R(Rm.T @ Ri.T @ Rj)
            Ji = jr_inv(rr)                       # exact, not the small-angle form
            wt = 1.0 / (VO_SIGMA_T * max(dt[k], 1e-3) / 0.1)
            wr = 1.0 / (VO_SIGMA_R * max(dt[k], 1e-3) / 0.1)
            Jt_i = np.hstack([-Ri.T, hat(Ri.T @ d)]) * wt
            Jt_j = np.hstack([Ri.T, np.zeros((3, 3))]) * wt
            Jr_i = np.hstack([np.zeros((3, 3)), -Ji @ Rj.T @ Ri]) * wr
            Jr_j = np.hstack([np.zeros((3, 3)), Ji]) * wr
            B = np.vstack([np.hstack([Jt_i, Jt_j]), np.hstack([Jr_i, Jr_j])])
            cols = list(range(6 * k, 6 * k + 6)) + list(range(6 * (k + 1), 6 * (k + 1) + 6))
            rows, cc = np.meshgrid(np.arange(6), np.arange(12), indexing="ij")
            vv = B.ravel(); cl = [cols[c] for c in cc.ravel()]
            if ns:
                vv = np.r_[vv, (-Zm[:3, 3] * wt), np.zeros(3)]
                cl = cl + [NS] * 6
                rows = np.r_[rows.ravel(), np.arange(6)]
            else:
                rows = rows.ravel()
            add(rows, cl, vv, list(np.r_[rt * wt, rr * wr]))
        # ---- scan-to-map, relinearised every iteration ----
        # NOT a pose prior built from the chained ICP pass. Those poses are correlated -
        # each is seeded from the previous - so feeding them in as independent absolute
        # measurements lets the chain's own drift accumulate as evidence. Measured here:
        # along a corridor that pseudo-evidence reached 2.3e6 of information, more than a
        # board contributes (1.9e6), and the boards lost. Point-to-plane residuals evaluated
        # at the current estimate have no seed in them at all.
        if use_icp:
            for k in range(n):
                ii = NODE_ICP[k]
                if ii < 0: continue
                P = CL[ii]
                if len(P) > ICP_PTS:
                    P = P[np.linspace(0, len(P) - 1, ICP_PTS).astype(int)]
                Q = apply(Ts[k], P)
                c, nn, w, m = REF.plane_of(Q)
                if m.sum() < 30: continue
                p = P[m]; R = Ts[k][:3, :3]
                r = np.einsum("ij,ij->i", Q[m] - c, nn)
                keep = np.abs(r) < MAX_CORR[-1]
                if keep.sum() < 30: continue
                c, nn, w, p, r = c[keep], nn[keep], w[keep], p[keep], r[keep]
                ww = w * np.minimum(1.0, HUBER / np.maximum(np.abs(r), 1e-9)) / ICP_SIGMA
                Jp = np.hstack([nn, np.cross(p, nn @ R)]) * ww[:, None]
                rows, cc = np.meshgrid(np.arange(len(r)), np.arange(6), indexing="ij")
                add(rows.ravel(), [6 * k + cix for cix in cc.ravel()],
                    Jp.ravel(), list(r * ww))
        # ---- board reprojection ----
        if use_board:
            for k, lst in NODE_SIGHT.items():
                if k >= n: continue
                for sg_ in lst:
                    if sg_["board"] not in boards_used: continue
                    b = BOARDS[sg_["board"]]
                    Pm = board_corners_map(sg_["board"], sg_["ids"])
                    uv = sg_["uv"]; f = 0.5 * (K_IR[0, 0] + K_IR[1, 1])
                    sig = math.sqrt(PIX_SIGMA ** 2 +
                                    (f * b["sigma_t"] / max(sg_["rng"], .3)) ** 2 +
                                    (f * b["sigma_r"] * 0.11 / max(sg_["rng"], .3)) ** 2)
                    def resid(T):
                        return (project(K_IR, apply(inv(T), Pm)) - uv).ravel()
                    r0 = resid(Ts[k])
                    hub = np.minimum(1.0, BOARD_HUBER / np.maximum(np.abs(r0), 1e-6))
                    w = hub / sig
                    Jn = np.zeros((len(r0), 6)); eps = 1e-6
                    for c in range(6):
                        d = np.zeros(6); d[c] = eps
                        Tp = Rt(Ts[k][:3, :3] @ exp_r(d[3:]), Ts[k][:3, 3] + d[:3])
                        Jn[:, c] = (resid(Tp) - r0) / eps
                    Jn *= w[:, None]
                    rows, cc = np.meshgrid(np.arange(len(r0)), np.arange(6), indexing="ij")
                    add(rows.ravel(), [6 * k + c for c in cc.ravel()], Jn.ravel(), list(r0 * w))
        Aj = sparse.csr_matrix((V_, (I_, J_)), shape=(len(r_), NS + ns))
        rv = np.array(r_)
        Hn = (Aj.T @ Aj).tocsc() + sparse.identity(NS + ns, format="csc") * 1e-6
        dx = spsolve(Hn, -(Aj.T @ rv))
        for k in range(n):
            d = dx[6 * k:6 * k + 6]
            Ts[k] = Rt(Ts[k][:3, :3] @ exp_r(d[3:]), Ts[k][:3, 3] + d[:3])
        if ns: s += dx[NS]
        step = np.linalg.norm(dx[:NS].reshape(-1, 6)[:, :3], axis=1)
        cost = float(rv @ rv)
        if verbose:
            print(f"  it{it:2d}  cost {cost:12.1f}  max step {step.max()*1000:7.2f} mm"
                  f"  scale {s:.5f}")
        rep = dict(cost=cost, max_step=float(step.max()), scale=float(s), iters=it + 1)
        if step.max() < 1e-5: break
    return Ts, s, rep
print("graph ready")

In [ ]:
# ============================== run the three arms ==============================
NODE_T, NODE_ICP, NODE_SIGHT = build_nodes()
print(f"{len(NODE_T)} nodes ({np.sum(NODE_ICP>=0)} with an ICP factor, "
      f"{len(NODE_SIGHT)} with sightings)\n")
ARMS = {}
print("== arm A (relative + ICP, no boards) ==")
ARMS["A"] = solve_graph(use_icp=True,  use_board=False)
print("\n== arm B (relative + boards, no ICP) ==")
ARMS["B"] = solve_graph(use_icp=False, use_board=True)
print("\n== arm C (relative + ICP + boards) ==")
ARMS["C"] = solve_graph(use_icp=True,  use_board=True)

for k, (Ts, s, rep) in ARMS.items():
    write_traj(WORK / f"traj_m2_arm{k}.txt", NODE_T, Ts)
    # s multiplies the MEASURED translation, so the physical VSLAM scale is 1/s
    print(f"arm {k}: VSLAM scale {1/s:.5f} ({(1/s-1)*1e6:+.0f} ppm)  "
          f"cost {rep['cost']:.1f}  iters {rep['iters']}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for k, (Ts, s, _) in ARMS.items():
    ax[0].plot(Ts[:, 0, 3], Ts[:, 1, 3], lw=1.2, label=f"arm {k}")
ax[0].scatter(REF.P[::200, 0], REF.P[::200, 1], s=.2, c="0.8", zorder=0)
for nm, b in BOARDS.items():
    p = b["T_map_board"][:3, 3]; ax[0].plot(*p[:2], "r*", ms=14)
    ax[0].annotate(nm, p[:2], fontsize=8)
ax[0].set_aspect("equal"); ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title("XY, map frame")
base = ARMS["C"][0]
for k in ("A", "B"):
    ax[1].plot(NODE_T - NODE_T[0],
               np.linalg.norm(ARMS[k][0][:, :3, 3] - base[:, :3, 3], axis=1) * 100,
               lw=1.0, label=f"|arm {k} - arm C|")
for _, a, z, _ in WINDOWS: ax[1].axvspan(a, z, color="g", alpha=.12)
ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("cm"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title("disagreement vs the joint arm (green = board in view)")
plt.tight_layout(); plt.show()

## Step 4 - evaluation

### Leave-one-board-out is not optional

If all three boards enter the estimate, there is no independent check on any arm and the
ablation table is self-referential. Estimate with two boards, score on the third, rotate. That
produces a number you can defend in review.

Arm A never uses boards, so *every* board is held out for it - which is exactly why arm A's
column is the honest baseline and why it should be reported at all three boards.

In [ ]:
# ============================== leave-one-board-out ==============================
def board_error(Ts, node_t, board):
    """Predicted vs surveyed board pose at each sighting of `board`. -> (mm array, deg array)"""
    et, er = [], []
    for s in SIGHT:
        if s["board"] != board: continue
        k = int(np.argmin(np.abs(node_t - s["t"])))
        if abs(node_t[k] - s["t"]) > 0.02: continue
        T_map_b_hat = Ts[k] @ s["T_cb"]                 # camera pose x PnP board pose
        T_srv = BOARDS[board]["T_map_board"]
        et.append(np.linalg.norm(T_map_b_hat[:3, 3] - T_srv[:3, 3]) * 1000)
        er.append(math.degrees(np.linalg.norm(log_R(T_srv[:3, :3].T @ T_map_b_hat[:3, :3]))))
    return np.array(et), np.array(er)

LOO = {}
for h in bnames:
    used = [b for b in BOARDS if b != h]
    print(f"\n--- holding out '{h}' (estimating with {used}) ---")
    LOO[h] = {}
    LOO[h]["A"] = ARMS["A"]                                        # never uses boards
    LOO[h]["B"] = solve_graph(False, True,  boards_used=used, verbose=False)
    LOO[h]["C"] = solve_graph(True,  True,  boards_used=used, verbose=False)
    for k in "ABC":
        et, er = board_error(LOO[h][k][0], NODE_T, h)
        if len(et):
            print(f"   arm {k}: held-out '{h}'  median {np.median(et):7.1f} mm / "
                  f"{np.median(er):5.2f} deg   p95 {np.percentile(et,95):7.1f} mm   n={len(et)}")
        else:
            print(f"   arm {k}: no node within 20 ms of a '{h}' sighting")

In [ ]:
# ============================== map-surface agreement ==============================
def surface_agreement(Ts, node_t, stride=5, min_pts=25):
    """Per-cell mean signed offset of the depth cloud from the map's local plane.
    |mean| = surface PLACEMENT (bias, what a pose error looks like);
    spread  = surface THICKNESS (what sensor noise looks like). They are different failures."""
    acc = {}
    for i in range(0, len(CL), stride):
        k = int(np.argmin(np.abs(node_t - CL_T[i])))
        Q = apply(Ts[k], CL[i])
        c, nn, w, m = REF.plane_of(Q)
        if m.sum() < 10: continue
        d = np.einsum("ij,ij->i", Q[m] - c, nn)
        q = REF._vox(Q[m]); key = (q[:, 0] << 40) | (q[:, 1] << 20) | q[:, 2]
        tilt = np.abs(nn[:, 2])
        for kk, dd, tt in zip(key, d, tilt):
            a = acc.setdefault(int(kk), [[], tt]); a[0].append(dd)
    off_w, off_f, spr = [], [], []
    for kk, (ds, tt) in acc.items():
        if len(ds) < min_pts: continue
        ds = np.array(ds); off = abs(float(np.mean(ds)))
        spr.append(np.percentile(ds, 95) - np.percentile(ds, 5))
        (off_w if tt < 0.5 else off_f).append(off)      # |n_z|<0.5 -> wall-like
    return (np.array(off_w), np.array(off_f), np.array(spr))

SURF = {}
for k, (Ts, s, _) in ARMS.items():
    w, f, sp = surface_agreement(Ts, NODE_T)
    SURF[k] = (w, f, sp)
    print(f"arm {k}: walls {np.median(w)*100:5.2f} cm ({len(w)} cells)   "
          f"floors {np.median(f)*100:5.2f} cm ({len(f)} cells)   "
          f"spread {np.median(sp)*100:5.2f} cm")
print("\nspread is the D455's own depth noise plus the map's ~2.9 cm wall floor - it does not "
      "shrink with a better pose. Only the placement numbers respond to the estimator.")

In [ ]:
# ============================== repeat-visit consistency ==============================
WINDOW_S, MIN_DT, MAX_DIST = 10.0, 60.0, 3.0
def repeat_visit(Ts, node_t):
    """Does the agent agree with itself when it comes back? Point-to-plane median between
    depth clouds from the same place at times separated by >= MIN_DT."""
    idx = np.arange(0, len(CL), 3)
    P = np.array([Ts[int(np.argmin(np.abs(node_t - CL_T[i])))][:3, 3] for i in idx])
    out = []
    for a in range(len(idx)):
        for b in range(a + 1, len(idx)):
            if CL_T[idx[b]] - CL_T[idx[a]] < MIN_DT: continue
            if np.linalg.norm(P[a] - P[b]) > MAX_DIST: continue
            Ta = Ts[int(np.argmin(np.abs(node_t - CL_T[idx[a]])))]
            Tb = Ts[int(np.argmin(np.abs(node_t - CL_T[idx[b]])))]
            Qa, Qb = apply(Ta, CL[idx[a]]), apply(Tb, CL[idx[b]])
            c, nn, w, m = REF.plane_of(Qa)
            if m.sum() < 200: continue
            da = np.einsum("ij,ij->i", Qa[m] - c, nn)
            c2, n2, w2, m2 = REF.plane_of(Qb)
            if m2.sum() < 200: continue
            db = np.einsum("ij,ij->i", Qb[m2] - c2, n2)
            out.append(abs(np.median(da) - np.median(db)))
            break
    return np.array(out)

for k, (Ts, s, _) in ARMS.items():
    rv = repeat_visit(Ts, NODE_T)
    print(f"arm {k}: repeat-visit disagreement "
          + (f"median {np.median(rv)*100:.2f} cm over {len(rv)} pairs" if len(rv)
             else "-- no revisits found (raise MAX_DIST or lower MIN_DT)"))

In [ ]:
# ============================== the ablation table ==============================
rows = []
for k in "ABC":
    et_all, er_all = [], []
    for h in bnames:
        et, er = board_error(LOO[h][k][0], NODE_T, h)
        et_all += list(et); er_all += list(er)
    w, f, sp = SURF[k]
    icp_ok = 100 * np.mean(A_NOBS == 6) if k in ("A", "C") else np.nan
    rows.append(dict(
        arm=k,
        loo_mm=np.median(et_all) if et_all else np.nan,
        loo_p95=np.percentile(et_all, 95) if et_all else np.nan,
        loo_deg=np.median(er_all) if er_all else np.nan,
        wall_cm=np.median(w) * 100, floor_cm=np.median(f) * 100,
        scale=ARMS[k][1], obs=icp_ok))

hdr = ("arm", "held-out board (mm)", "p95 (mm)", "(deg)", "wall place (cm)",
       "floor place (cm)", "VO scale", "6-DoF frames (%)")
print(f"{hdr[0]:4s}{hdr[1]:>21s}{hdr[2]:>10s}{hdr[3]:>8s}{hdr[4]:>17s}"
      f"{hdr[5]:>18s}{hdr[6]:>10s}{hdr[7]:>18s}")
for r in rows:
    print(f"{r['arm']:4s}{r['loo_mm']:21.1f}{r['loo_p95']:10.1f}{r['loo_deg']:8.2f}"
          f"{r['wall_cm']:17.2f}{r['floor_cm']:18.2f}{r['scale']:10.5f}"
          f"{r['obs']:18.1f}")

longest = max(w[2] - w[1] for w in WINDOWS) if WINDOWS else 0
print(f"""
Reading the table
  arm A  geometry only. Holds the pose where the frustum sees structure, drifts where it does not.
         Its held-out error is the honest board-free number; a large p95 against the median is the
         degenerate corridor stretches showing up.
  arm B  boards + VSLAM. Tight at sightings, worst in the longest board-free stretch
         ({longest:.0f} s here). Rotation is its weak column - a small planar target constrains
         out-of-plane rotation poorly, so between anchors this is VSLAM rotation drift.
  arm C  both. C should beat A on the MAX and match it on the median and on rotation, while
         beating B everywhere except right at a board. If C is worse than either, a weight is
         wrong - the first suspect is a board sigma, the second is VO_SIGMA_T.

  On the synthetic corridor this stack was validated against (see the self-test cell), the arms
  came out A 1.55/8.14 cm, B 4.11/13.55 cm, C 1.49/5.31 cm (median/max), rotation 0.15 / 0.96 /
  0.15 deg. Board-derived VSLAM scale was good to 200 ppm where geometry alone managed 2600 ppm -
  long absolute baselines beat local structure for scale, which is the argument for the boards
  independent of anything else.

Caveats to carry into the paper
  - board poses come from a survey 96 min earlier; step 0c is a joint test of board stability and
    VSLAM scale and cannot separate them
  - {100*np.mean(A_NOBS<6):.0f}% of frames are rank-deficient; those poses are supported by the
    graph, not by their own geometry
  - the spread column of surface_agreement is D455 depth noise, not pose error - do not quote it
    as an accuracy figure
""")

for k in "ABC":
    write_traj(WORK / f"traj_m2_arm{k}.txt", NODE_T, ARMS[k][0])
print("reference trajectories written to", WORK.resolve())

---
## Appendix - self-test on synthetic ground truth

Runs the whole estimator stack against a synthetic 24 m corridor where the true trajectory is
known, so you can tell a broken pipeline from bad data. **No bag, map or survey needed** - run the
CONFIG and IMPORTS cells, the four `... ready` definition cells (charuco helpers, `Reference`,
cloud helpers, `icp_frame`, `solve_graph`), then this. It overwrites the pipeline globals, so use a
fresh kernel afterwards.

The scene is chosen to reproduce the real failure mode: a 3.5 m range gate means the end walls
drop out mid-corridor, leaving two parallel walls, a floor and a ceiling - which do not constrain
the along-corridor direction. Boards sit on the two end walls and one angled side-wall mount.

Expected output (this is what the stack produced when it was validated):

```
per-frame ICP at ground truth   0.29 cm            <- frame-level noise floor
arm A  geometry only   med 1.55  max  8.14 cm   rot 0.148 deg
arm B  boards + VSLAM  med 4.11  max 13.55 cm   rot 0.955 deg
arm C  joint           med 1.49  max  5.31 cm   rot 0.149 deg
```

What each row is telling you, and what to do if yours disagrees:
- **A's max >> A's median** - correct. That is the degenerate stretch, and it is the whole reason
  the boards are in the design.
- **B's rotation ~6x worse than A's** - correct. An 18 cm planar target constrains out-of-plane
  rotation weakly, so between sightings this is VSLAM rotation drift, growing as sqrt(n).
- **C beats A on max and ties it on median and rotation** - the claim of the method. If C is worse
  than A *or* than B, a weight is wrong: suspect a board sigma first, `VO_SIGMA_T` second.
- **A's ATE far above 0.29 cm** - `VO_SIGMA_T` is too loose. This one number moved ATE from
  0.27 cm to 2.47 cm during validation; measure it, do not guess it.

In [ ]:
# ============================== SELF-TEST (no bag needed) ==============================
RUN_SELFTEST = True
SELFTEST_SEED = 7
if RUN_SELFTEST:
    _rng = np.random.default_rng(SELFTEST_SEED)
    def _plane(o, u, v, nu, nv):
        a, b = np.meshgrid(np.linspace(0, 1, nu), np.linspace(0, 1, nv), indexing="ij")
        return np.asarray(o) + a.ravel()[:, None] * np.asarray(u) + b.ravel()[:, None] * np.asarray(v)
    _L, _W, _H = 24.0, 2.4, 2.8
    _S = [_plane([0,0,0],[_L,0,0],[0,0,_H],240,30), _plane([0,_W,0],[_L,0,0],[0,0,_H],240,30),
          _plane([0,0,0],[_L,0,0],[0,_W,0],240,26), _plane([0,0,_H],[_L,0,0],[0,_W,0],240,26),
          _plane([0,-2,0],[0,_W+4,0],[0,0,_H],40,30), _plane([_L,-2,0],[0,_W+4,0],[0,0,_H],40,30)]
    _MAP = np.vstack(_S) + _rng.normal(scale=0.010, size=(sum(len(s) for s in _S), 3))

    _N = 180
    _x = np.r_[np.linspace(2, _L-2, _N//2), np.linspace(_L-2, 2, _N//2)]
    _yaw = np.r_[np.zeros(_N//2), np.full(_N//2, np.pi)]
    _Rbc = np.array([[0.,0.,1.], [-1.,0.,0.], [0.,-1.,0.]])          # body -> camera optical
    GT = np.array([Rt(Rot.from_euler('z', _yaw[i]).as_matrix() @ _Rbc,
                      np.array([_x[i], _W/2 + 0.05*np.sin(i/7), 1.05])) for i in range(_N)])
    TS = np.arange(_N) * 0.1
    _K = np.array([[425.,0.,424.], [0.,425.,240.], [0.,0.,1.]])

    def _render(T):
        P = (_MAP - T[:3,3]) @ T[:3,:3]; z = P[:,2]
        ok = (z > RANGE_MIN) & (z < RANGE_MAX)
        u = _K[0,0]*P[:,0]/np.where(z>0,z,1) + _K[0,2]
        v = _K[1,1]*P[:,1]/np.where(z>0,z,1) + _K[1,2]
        ok &= (u>0)&(u<848)&(v>0)&(v<480)
        Q = P[ok]
        return (Q + _rng.normal(scale=0.005+0.008*(Q[:,2:3]/RANGE_MAX)**2, size=Q.shape)).astype(np.float32)

    CL = [_render(T) for T in GT]; CL_T = TS.copy()
    REF = Reference(_MAP); K_IR = _K

    _s_true = 1.004                                   # VSLAM stereo scale error
    _vo = [np.eye(4)]
    for i in range(1, _N):
        Z = inv(GT[i-1]) @ GT[i]
        _vo.append(_vo[-1] @ Rt(Z[:3,:3] @ Rot.from_rotvec(_rng.normal(scale=0.0015,size=3)).as_matrix(),
                                Z[:3,3]*_s_true + _rng.normal(scale=0.004,size=3)))
    vo_t, vo_T = TS.copy(), np.array(_vo)
    X = np.eye(4); T_map_vo = GT[0]

    _BASE = np.array([[0.,0.,1.], [-1.,0.,0.], [0.,-1.,0.]])   # board normal (+Z_cv) along +x
    BOARDS, AXIS_CANDIDATES, BOARD_AXES = {}, {"cv": np.eye(3)}, "cv"
    for nm, (px,py,pz,ang,st,sr) in {
            "anchor":   (0.05, 1.20, 1.2, 0.0,        0.0069, 0.51),
            "anchor_b": (12.0, 0.02, 1.2, 3*np.pi/4,  0.0129, 2.64),
            "rs_anchor":(23.95,1.20, 1.2, np.pi,      0.0181, 0.61)}.items():
        BOARDS[nm] = dict(name=nm, squares=(9,7), square_len=0.02, dictionary="DICT_4X4_50",
                          id_offset=0, sigma_t=st, sigma_r=math.radians(sr),
                          T_map_board=Rt(Rot.from_euler('z', ang).as_matrix() @ _BASE,
                                         np.array([px,py,pz])))
    SIGHT = []
    for i, T in enumerate(GT):
        for nm, b in BOARDS.items():
            Tb = b["T_map_board"]; Pm = apply(Tb, board_object_points(b)); Pc = apply(inv(T), Pm)
            if Pc[:,2].min() < 0.4 or Pc[:,2].max() > 2.0: continue
            tc = T[:3,3] - Tb[:3,3]; tc /= np.linalg.norm(tc)
            if float(Tb[:3,2] @ tc) < math.cos(math.radians(60)): continue   # front face only
            uv = project(_K, Pc)
            if uv[:,0].min()<10 or uv[:,0].max()>838 or uv[:,1].min()<10 or uv[:,1].max()>470: continue
            SIGHT.append(dict(t=TS[i], board=nm, rng=float(np.linalg.norm(Pc.mean(0))),
                              ids=np.arange(48), uv=uv + _rng.normal(scale=0.5, size=uv.shape),
                              T_cb=inv(T) @ Tb))
    for nm, b in BOARDS.items():                       # inject the survey error the sigmas claim
        b["T_map_board"] = b["T_map_board"] @ Rt(
            Rot.from_rotvec(_rng.normal(scale=b["sigma_r"], size=3)).as_matrix(),
            _rng.normal(scale=b["sigma_t"], size=3))
    import collections
    print("sightings:", dict(collections.Counter(s["board"] for s in SIGHT)))

    _e = [np.linalg.norm(icp_frame(CL[i], GT[i], REF)[0][:3,3] - GT[i][:3,3]) for i in range(0,_N,10)]
    print(f"per-frame ICP at ground truth   {np.median(_e)*100:.2f} cm   <- frame-level noise floor")

    _T = GT[0].copy()
    A_T = np.zeros((_N,4,4)); A_INFO = np.zeros((_N,6,6))
    A_RMS = np.zeros(_N); A_NOBS = np.zeros(_N,int); A_EIG = np.zeros((_N,6))
    for i in range(_N):
        if i: _T = _T @ (inv(vo_T[i-1]) @ vo_T[i])
        A_T[i], A_INFO[i], _, A_RMS[i], A_EIG[i], A_NOBS[i] = icp_frame(CL[i], _T, REF)
        _T = A_T[i]
    print("observable DOF:", dict(sorted(collections.Counter(A_NOBS.tolist()).items())))

    NODE_T, NODE_ICP, NODE_SIGHT = build_nodes()
    WINDOWS = []
    for _nm, (_ui, _ub) in {"A  geometry only":(True,False), "B  boards + VSLAM":(False,True),
                            "C  joint":(True,True)}.items():
        _Ts, _s, _ = solve_graph(use_icp=_ui, use_board=_ub, verbose=False)
        _Ti = interp_traj(NODE_T, _Ts, TS)
        _et = np.linalg.norm(_Ti[:,:3,3] - GT[:,:3,3], axis=1)
        _er = np.array([math.degrees(np.linalg.norm(log_R(GT[i][:3,:3].T @ _Ti[i][:3,:3])))
                        for i in range(_N)])
        _sight = np.array([np.min(np.abs(TS[i] - [s["t"] for s in SIGHT])) for i in range(_N)])
        _far = _sight > 3.0                       # frames >3 s from any sighting
        print(f"arm {_nm:20s} med {np.median(_et)*100:5.2f}  max {np.max(_et)*100:6.2f} cm   "
              f"rot {np.median(_er):.3f} deg   near-board {np.median(_et[~_far])*100:5.2f} cm   "
              f"scale {1/_s:.5f}")
    print("\ncompare against the expected block in the markdown above. "
          "Restart the kernel before running the real pipeline.")